Instalações

In [2]:
!pip install opencv-python
!pip install tensorflow

Import


In [3]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    BatchNormalization,
    GlobalAveragePooling2D
)

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from tensorflow.keras.preprocessing.image import ImageDataGenerator


from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

from tensorflow.keras.optimizers import Adam

import kagglehub

Baixando o Dataset

In [4]:
path = kagglehub.dataset_download(
    "andrewmvd/ocular-disease-recognition-odir5k"
)

dados = pd.read_csv(
    os.path.join(path, "full_df.csv")
)

100%|██████████| 1.62G/1.62G [00:32<00:00, 53.7MB/s]

Extracting files...


Limpeza: Removendo os espaços extras dos nomes

In [5]:
dados["Left-Fundus"] = dados["Left-Fundus"].astype(str).str.strip()
dados["Right-Fundus"] = dados["Right-Fundus"].astype(str).str.strip()

Função para verificar se a rotulagem é catarata ou não

In [6]:
def possui_catarata(texto):
    texto = str(texto).lower()
    if "cataract" in texto:
        return 1
    return 0

Separação entre catarata e normal

In [7]:
dados["catarata_esquerda"] = dados["Left-Diagnostic Keywords"].apply(possui_catarata)
dados["catarata_direita"] = dados["Right-Diagnostic Keywords"].apply(possui_catarata)

catarata_esquerda = dados.loc[
    (dados.C == 1) & (dados.catarata_esquerda == 1)
]["Left-Fundus"].values

catarata_direita = dados.loc[
    (dados.C == 1) & (dados.catarata_direita == 1)
]["Right-Fundus"].values

normal_esquerda = dados.loc[
    dados["Left-Diagnostic Keywords"] == "normal fundus"
]["Left-Fundus"].values

normal_direita = dados.loc[
    dados["Right-Diagnostic Keywords"] == "normal fundus"
]["Right-Fundus"].values

imagens_catarata = np.concatenate((catarata_esquerda, catarata_direita))
imagens_normais = np.concatenate((normal_esquerda, normal_direita))

print("Catarata:", len(imagens_catarata))
print("Normais:", len(imagens_normais))


Catarata: 594
Normais: 5501


Criando o DataFrame

In [8]:
PASTA_IMAGENS = os.path.join(path, "preprocessed_images")

df_catarata = pd.DataFrame({"filename": imagens_catarata, "label": "catarata"})
df_normais = pd.DataFrame({"filename": imagens_normais, "label": "normal"})

df_total = pd.concat([df_catarata, df_normais], ignore_index=True)

def arquivo_existe(nome):
    return os.path.exists(os.path.join(PASTA_IMAGENS, str(nome).strip()))

df_total["existe"] = df_total["filename"].apply(arquivo_existe)
df_total = df_total[df_total["existe"] == True].drop(columns=["existe"])

print(f"Total de imagens válidas para o treino/teste: {len(df_total)}")

Total de imagens válidas para o treino/teste: 6089


Ajustes e processamento dos dados para treino e teste

In [9]:
df_treino, df_teste = train_test_split(
    df_total,
    test_size=0.20,
    stratify=df_total["label"],
    random_state=42
)
gerador_treino_config = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    horizontal_flip=True
)

gerador_teste_config = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

TAMANHO = 224
BATCH = 32

Preparação dos lotes de treino e de teste

In [10]:

gerador_treino = gerador_treino_config.flow_from_dataframe(
    dataframe=df_treino,
    directory=PASTA_IMAGENS,
    x_col="filename",
    y_col="label",
    target_size=(TAMANHO, TAMANHO),
    batch_size=BATCH,
    class_mode="binary"
)

gerador_teste = gerador_teste_config.flow_from_dataframe(
    dataframe=df_teste,
    directory=PASTA_IMAGENS,
    x_col="filename",
    y_col="label",
    target_size=(TAMANHO, TAMANHO),
    batch_size=BATCH,
    class_mode="binary",
    shuffle=False
)

Found 4871 validated image filenames belonging to 2 classes.
Found 1218 validated image filenames belonging to 2 classes.


Começa o modelo

In [11]:
base_modelo = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(TAMANHO, TAMANHO, 3)
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Selecionando camadas de treinamento

In [12]:
for camada in base_modelo.layers[:-30]:
    camada.trainable = False

for camada in base_modelo.layers[-30:]:
    camada.trainable = True

Criando o modelo

In [13]:
modelo = Sequential([
    base_modelo,

    GlobalAveragePooling2D(),

    Dense(512, activation="relu"),
    BatchNormalization(),
    Dropout(0.5),

    Dense(256, activation="relu"),
    BatchNormalization(),
    Dropout(0.3),

    Dense(1, activation="sigmoid")
])

modelo.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

modelo.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,048,513 (11.63 MB)

 Trainable params: 2,315,393 (8.83 MB)

 Non-trainable params: 733,120 (2.80 MB)

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    min_lr=1e-6
)

checkpoint = ModelCheckpoint(
    "modelo_catarata_mobilenetv2.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

historico = modelo.fit(
    gerador_treino,
    validation_data=gerador_teste,
    epochs=30,
    callbacks=[early_stop, reduce_lr, checkpoint]
)

Epoch 1/30
153/153 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.6755 - loss: 0.7297
Epoch 1: val_accuracy improved from None to 0.77750, saving model to modelo_catarata_mobilenetv2.keras

Epoch 1: finished saving model to modelo_catarata_mobilenetv2.keras
153/153 ━━━━━━━━━━━━━━━━━━━━ 504s 3s/step - accuracy: 0.7463 - loss: 0.6039 - val_accuracy: 0.7775 - val_loss: 0.5500 - learning_rate: 1.0000e-04
Epoch 2/30
153/153 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8875 - loss: 0.3801
Epoch 2: val_accuracy improved from 0.77750 to 0.85057, saving model to modelo_catarata_mobilenetv2.keras

Epoch 2: finished saving model to modelo_catarata_mobilenetv2.keras
153/153 ━━━━━━━━━━━━━━━━━━━━ 486s 3s/step - accuracy: 0.8957 - loss: 0.3639 - val_accuracy: 0.8506 - val_loss: 0.4588 - learning_rate: 1.0000e-04
Epoch 3/30
153/153 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9410 - loss: 0.2622
Epoch 3: val_accuracy improved from 0.85057 to 0.96223, saving model to modelo_catarata_mobilenetv2.keras


Avaliação do modelo

In [ ]:
print("\nAvaliando modelo...")
perda, acuracia = modelo.evaluate(gerador_teste)

print("\nAcurácia:", acuracia)
print("Perda:", perda)

probabilidades = modelo.predict(gerador_teste)
predicoes = (probabilidades > 0.5).astype(int)

y_teste_real = gerador_teste.classes


print("\nRelatório de Classificação:")
print(classification_report(y_teste_real, predicoes, target_names=["Catarata", "Normal"]))

print("Precision:", precision_score(y_teste_real, predicoes))
print("Recall:", recall_score(y_teste_real, predicoes))
print("F1 Score:", f1_score(y_teste_real, predicoes))

matriz = confusion_matrix(y_teste_real, predicoes)

plt.figure(figsize=(8,6))
plt.imshow(matriz, cmap='Blues')
plt.colorbar()
classes_inferidas = list(gerador_teste.class_indices.keys())

plt.xticks([0,1], classes_inferidas)
plt.yticks([0,1], classes_inferidas)

for i in range(2):
    for j in range(2):
        plt.text(
            j, i, matriz[i,j],
            ha="center", va="center",
            color="white" if matriz[i,j] > matriz.max()/2 else "black"
        )

plt.xlabel("Previsto")
plt.ylabel("Real")
plt.title("Matriz de Confusão")
plt.show()

modelo.save("modelo_catarata_mobilenetv2_final.keras")
print("\nModelo salvo com sucesso!")